# Elo Diff Comparison

Confronto minimale tra differenza Elo lineare e differenza Elo logaritmica, con baseline semplice e LightGBM a una sola feature.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

import lightgbm as lgb


In [2]:
PROCESSED_DIR = Path("../data/processed")

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")


In [3]:
train_df["elo_log_diff"] = np.log(train_df["player_elo"]) - np.log(train_df["opponent_elo"])
test_df["elo_log_diff"] = np.log(test_df["player_elo"]) - np.log(test_df["opponent_elo"])

TARGET_COL = "target"


In [4]:
def evaluate_single_feature(feature_col, baseline_fn):
    X_train = train_df[[feature_col]].copy()
    y_train = train_df[TARGET_COL]

    X_test = test_df[[feature_col]].copy()
    y_test = test_df[TARGET_COL]

    fill_value = X_train[feature_col].median()
    X_train[feature_col] = X_train[feature_col].fillna(fill_value)
    X_test[feature_col] = X_test[feature_col].fillna(fill_value)

    model = lgb.LGBMClassifier(
        objective="binary",
        metric="binary_logloss",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=7,
        min_child_samples=30,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(50)
        ]
    )

    test_pred_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_pred_proba >= 0.5).astype(int)

    baseline_score = X_test[feature_col]
    baseline_proba = baseline_fn(baseline_score)
    baseline_pred = (baseline_proba >= 0.5).astype(int)

    return {
        "feature": feature_col,
        "baseline_accuracy": accuracy_score(y_test, baseline_pred),
        "baseline_log_loss": log_loss(y_test, baseline_proba),
        "baseline_auc": roc_auc_score(y_test, baseline_proba),
        "lgbm_accuracy": accuracy_score(y_test, test_pred),
        "lgbm_log_loss": log_loss(y_test, test_pred_proba),
        "lgbm_auc": roc_auc_score(y_test, test_pred_proba),
    }


In [5]:
linear_results = evaluate_single_feature(
    "elo_diff",
    baseline_fn=lambda score: 1 / (1 + np.power(10.0, -score / 400.0)),
)

log_results = evaluate_single_feature(
    "elo_log_diff",
    baseline_fn=lambda score: 1 / (1 + np.exp(-score)),
)


[LightGBM] [Info] Number of positive: 114146, number of negative: 114146
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000535 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 228292, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 30 rounds
[50]	valid_0's binary_logloss: 0.622902
[100]	valid_0's binary_logloss: 0.617811
[150]	valid_0's binary_logloss: 0.617386
[200]	valid_0's binary_logloss: 0.617346
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[250]	valid_0's binary_logloss: 0.617337
[LightGBM] [Warning] No further splits with positive gain, best gain: 

In [6]:
pd.DataFrame([
    linear_results,
    log_results,
]).set_index("feature")


,baseline_accuracy,baseline_log_loss,baseline_auc,lgbm_accuracy,lgbm_log_loss,lgbm_auc
feature,,,,,,
elo_diff,0.648809,0.61998,0.713239,0.649223,0.617332,0.713264
elo_log_diff,0.648809,0.67694,0.713057,0.649223,0.618053,0.712884
